In [80]:
import numpy as np
import pandas as pd
import random
import time
from rapidfuzz import process, fuzz, distance
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import spoa # Ensure pip install spoa

# --- 1. CONFIGURATION ---
seed = 42
random.seed(seed)
np.random.seed(seed)

# --- 2. DATA GENERATION (Your Code) ---
# print("--- Generating Data ---")

def mutate_sequence(seq, error_rate=0.10):
    if error_rate <= 0: return seq
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            r = random.random()
            if r < 0.5: new_seq.append(random.choice("ACGT")) 
            elif r < 0.75: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT"))
            else: pass 
        else:
            new_seq.append(base)
    return "".join(new_seq)

def gen_dna(k): return "".join(random.choices("ACGT", k=k))

# --- 3. CORE ALGORITHM FUNCTIONS ---

def encode_msa(msa_strings, alphabet="ACGTN-"):
    """Converts MSA strings to integer matrix."""
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    # Pad to max length just in case SPOA returns jagged list (unlikely with internal padding)
    max_len = max(len(s) for s in msa_strings)
    msa_padded = [s.ljust(max_len, '-') for s in msa_strings]
    
    msa_array = np.array([list(seq) for seq in msa_padded])
    N, L = msa_array.shape
    msa_int = np.zeros((N, L), dtype=np.int8)
    for char, idx in char_to_int.items():
        msa_int[msa_array == char] = idx
    return msa_int, len(alphabet)

def get_marginals(msa_int, vocab_size):
    one_hot = np.eye(vocab_size)[msa_int]
    P_i = one_hot.mean(axis=0)
    return one_hot, P_i

def compute_mi_scores(one_hot, P_i):
    """Calculates Mutual Information Matrix."""
    N, L, A = one_hot.shape
    flat_view = one_hot.transpose(1, 2, 0).reshape(L * A, N)
    joint_probs_flat = (flat_view @ flat_view.T) / N
    P_ij = joint_probs_flat.reshape(L, A, L, A)
    P_product = P_i[:, :, None, None] * P_i[None, None, :, :]
    
    mask = P_ij > 0
    mi_matrix = np.zeros_like(P_ij)
    mi_matrix[mask] = P_ij[mask] * np.log(P_ij[mask] / P_product[mask])
    return mi_matrix.sum(axis=(1, 3))

def perform_pca_split_robust(msa_int, vocab_size, col_indices):
    """Option 1: Checks Variance Explained (PC1) for Low N."""
    N = msa_int.shape[0]
    msa_subset = msa_int[:, col_indices]
    subset_one_hot = np.eye(vocab_size)[msa_subset]
    features = subset_one_hot.reshape(N, -1)
    
    n_comps = min(N, 2)
    if n_comps < 2: return np.zeros(N), 0.0 # Cannot split N=1
    
    pca = PCA(n_components=n_comps)
    coords = pca.fit_transform(features)
    
    # --- Option 1 Check ---
    explained_var = pca.explained_variance_ratio_[0]
    
    if N < 5:
        # Strict low-N check
        if explained_var < 0.75: # If PC1 isn't dominant, it's noise
            return np.zeros(N), explained_var
        labels = (coords[:, 0] > 0).astype(int)
        score = explained_var
    else:
        # Standard GMM for larger groups
        try:
            gmm = GaussianMixture(n_components=2, random_state=42)
            labels = gmm.fit_predict(coords)
            if len(np.unique(labels)) < 2: 
                score = 0
            else: 
                score = silhouette_score(coords, labels)
        except:
            labels = (coords[:, 0] > 0).astype(int)
            score = explained_var

    return labels, score


def validate_split_percentile(cluster_A_indices, cluster_B_indices, all_df, local_df, barcode_msa_string_length, 
                              percentile_th=5, n_background_target=1000):
    """
    Option 2 (Percentile Method): 
    Compares the 'Split Distance' against a distribution of 'Random Distances'.
    
    Logic: If Cluster A and B are truly different plasmids, their distance should 
    be comparable to the distance between Cluster A and a random background read.
    If the split distance is smaller than the Nth percentile of random distances,
    it's likely just sequencing noise -> Merge.
    """
    # 1. Get sequences
    inserts_A = local_df.loc[cluster_A_indices, 'Insert'].tolist()
    inserts_B = local_df.loc[cluster_B_indices, 'Insert'].tolist()
    
    if not inserts_A or not inserts_B: return False

    # 2. Calculate 'Signal' (Distance between the two proposed clusters)
    #    We use mean pairwise distance.
    #    Note: 100 - fuzz.ratio = distance
    signal_dist = 100 - process.cdist(inserts_A, inserts_B, scorer=fuzz.ratio).mean()
    
    # 3. Generate 'Background Distribution' (Distance to 100 random reads)
    #    Select 100 reads from outside this cluster
    remaining_df = all_df.drop(local_df.index, errors='ignore')
    n_background = min(n_background_target, len(remaining_df))
    
    if n_background < 10: 
        # Fallback if dataset is tiny: just assume valid if distance is significant (>15%)
        # print(f"      [Opt2] Warning: Low background (N={n_background}). using raw threshold.")
        return signal_dist > 15.0

    random_inserts = remaining_df.sample(n=n_background)['Insert'].tolist()
    
    # Calculate distances from Cluster A to these 100 randoms
    # process.cdist returns a matrix; we want the mean distance to each random read
    bg_dists_A = 100 - process.cdist(inserts_A, random_inserts, scorer=fuzz.ratio)
    bg_dists_B = 100 - process.cdist(inserts_B, random_inserts, scorer=fuzz.ratio)
    
    # 4. The Percentile Check
    #    We check if our signal is at least in the 'percentile_th' (e.g., 5th) 
    #    of the background distribution.
    threshold_value = (np.percentile(bg_dists_A, percentile_th) + np.percentile(bg_dists_B, percentile_th))/2
    
    is_valid = signal_dist > threshold_value  # distance between insert seqs in proposed clusters are more dissimilar that would expect by random chance
    
    # print(f"      [Opt2 Percentile] Signal Dist: {signal_dist:.1f} | BG {percentile_th}th %ile: {threshold_value:.1f} | Valid: {is_valid}")
    return is_valid

def recursive_analysis(sub_df, all_df, barcode_target, depth=0, OPT1_THRESHOLD=0.8):
    """Recursive driver for MSA -> Split -> Validate."""

    # FIX 1: Correctly check for the presence of the target barcode
    # Note: Renamed 'barcode' parameter to 'barcode_target' for clarity
    if barcode_target not in set(sub_df['Barcode']):
        # If the target barcode that initiated this whole process isn't in this cluster, skip.
        # This prevents the recursion from running forever on decoys.
        # print(f"{'  ' * depth}-> [SKIP] Target barcode not present in this sub-cluster.")
        return []
    
    # Stop condition
    if len(sub_df) < 2: return [sub_df]
    
    # FIX 2: Define prefix correctly (The issue was argument mismatch, not this line itself)
    prefix = "  " * depth
    # print(f"{prefix}Analyzing cluster of {len(sub_df)} reads...")

    # A. MSA (Barcode + Spacer + Insert for linkage)
    # ... (Your MSA logic remains here) ...
    barcodes = sub_df['Barcode'].tolist()
    _, msa_barcodes = spoa.poa(barcodes, algorithm=2) 
    
    inserts = sub_df['Insert'].tolist()
    _, msa_inserts = spoa.poa(inserts, algorithm=2)
    
    spacer = "--------"

    barcode_msa_string_length = len(msa_barcodes[0] + spacer)
    print(barcode_msa_string_length)
    
    msa_strings = [b + spacer + i for b, i in zip(msa_barcodes, msa_inserts)]
    
    msa_int, vocab_size = encode_msa(msa_strings)
    
    # B. Feature Selection (MI)
    _, P_i = get_marginals(msa_int, vocab_size)
    one_hot = np.eye(vocab_size)[msa_int]
    mi_matrix = compute_mi_scores(one_hot, P_i)
    
    col_scores = mi_matrix.sum(axis=0)
    n_cols = min(50, max(5, int(len(col_scores)*0.1)))
    top_cols = np.argsort(col_scores)[::-1][:n_cols]

    # C. PCA & Split Proposal
    labels, score = perform_pca_split_robust(msa_int, vocab_size, top_cols)
    
    if len(np.unique(labels)) < 2:
        # print(f"{prefix}-> [DECISION: MERGE] No valid split found. (Score/Var: {score:.2f})")
        return [sub_df]
    
    # D. Validation Logic
    indices_A = sub_df.index[labels == 0]
    indices_B = sub_df.index[labels == 1]

    if score > OPT1_THRESHOLD:
        # print(f"{prefix}-> [DECISION: SPLIT] Method: Option 1 (Strong Signal). Score: {score:.2f} > {OPT1_THRESHOLD}")
        valid = True
    else:
        # print(f"{prefix}-> [CHECKING] Option 1 score weak ({score:.2f}). Running Percentile Check...")
        valid = validate_split_percentile(indices_A, indices_B, all_df, sub_df, barcode_msa_string_length, percentile_th=5)

    if valid:
        # if score <= OPT1_THRESHOLD:
            # print(f"{prefix}-> [DECISION: SPLIT] Method: Option 2 (Background Ratio Passed).")
            
        df_A = sub_df.loc[indices_A]
        df_B = sub_df.loc[indices_B]
        
        # FIX 3: Corrected Recursive Call Arguments
        # Pass all positional arguments and the keyword argument
        return (recursive_analysis(df_A, all_df, barcode_target, depth + 1, OPT1_THRESHOLD) + 
                recursive_analysis(df_B, all_df, barcode_target, depth + 1, OPT1_THRESHOLD))
    else:
        # print(f"{prefix}-> [DECISION: MERGE] Option 2 Failed (Indistinguishable from random noise).")
        return [sub_df]





# print("\n--- Starting Pipeline ---")

n_items = 50 # Reduced for demo speed (increase to 2000 for real run)
pool_bc = [gen_dna(5) for _ in range(n_items)] # Increased BC length slightly for realism
pool_ins = [gen_dna(50) for _ in range(n_items)]

n_repeat_bc = 10
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(50) for _ in range(n_repeat_bc)]

data = []
for i in range(n_items):
    n_reads = random.randint(3, 30) 
    for _ in range(n_reads):
        data.append({
            "ID": i,
            "Barcode": mutate_sequence(pool_bc[i], 0.05), 
            "Insert": mutate_sequence(pool_ins[i], 0.05)
        })

df = pd.DataFrame(data)
# print(f"Dataset: {len(df)} reads (Target Clusters: {n_items})")

barcode_counts = df['Barcode'].value_counts()
all_barcodes = np.array(barcode_counts.index.tolist())

# Process just the top barcode for demonstration
processed_indices = set()

# Iterating top 3 just to see if we catch decoys or multiple clusters
for i, (top_bc, count) in enumerate(barcode_counts.head(100).items()):
    
    # Check if this barcode was already consumed by a previous fuzzy group
    # (In a real app, you'd track indices, here we just skip if exact match logic)
    # For robust production code: maintain a set of 'assigned_read_ids'
    
    # print(f"\nProcessing Top Barcode #{i+1}: {top_bc} (Count: {count})")
    
    # 1. Fast Fuzzy Filter (The Funnel)
    # Find all barcodes in the universe similar to this top one
    scores = process.cdist([top_bc], all_barcodes, scorer=fuzz.ratio, dtype=np.uint8)[0]
    indices_in_range = np.where(scores > 85)[0] # 85% similarity threshold
    
    # Add Decoys (Logic from your prompt)
    if len(all_barcodes) > 10:
        decoy_bcs = np.random.choice(all_barcodes, 5, replace=False)
        # Ensure we don't accidentally pick the real ones
        candidate_bcs = np.concatenate((all_barcodes[indices_in_range], decoy_bcs))
    else:
        candidate_bcs = all_barcodes[indices_in_range]
        
    filtered_df = df[df['Barcode'].isin(candidate_bcs)].copy()
    
    if filtered_df.empty: continue
        
    # print(f"  > RapidFuzz gathered {len(filtered_df)} reads (including potential decoys).")
    
    # 2. Run the Deep Analysis
    final_clusters = recursive_analysis(filtered_df, df, top_bc)
    
    # print(f"  > Result: {len(final_clusters)} distinct clusters found.")
    for idx, c in enumerate(final_clusters):
        print(f"    Cluster {idx}: {len(c)} reads. (IDs: {c['ID'].unique()})")
        assert len(c['ID'].unique()) == 1

20
6
20
12
10
7
    Cluster 0: 14 reads. (IDs: [26])
    Cluster 1: 19 reads. (IDs: [27])
11
10
10
10
11
10
6
    Cluster 0: 1 reads. (IDs: [20])
    Cluster 1: 28 reads. (IDs: [29])
20
5
    Cluster 0: 26 reads. (IDs: [40])
14
14
6
    Cluster 0: 27 reads. (IDs: [31])
19
11
7
    Cluster 0: 26 reads. (IDs: [25])
19
11
6
    Cluster 0: 25 reads. (IDs: [37])
15
6
    Cluster 0: 25 reads. (IDs: [0])
13
6
    Cluster 0: 25 reads. (IDs: [7])
16
14
    Cluster 0: 31 reads. (IDs: [33 49])


AssertionError: 

In [79]:
print(top_bc)
print(final_clusters)

GACGC
[     ID Barcode                                             Insert
541  33   AAGAA  ATTACGAGAGGGACGAAGAGTCGCACTGCTGGACATTATACTTTGC...
542  33   AAGAA  ATTACGAGAGGGACGAAGAGATCGCACTGCTGGACATTATACTTTG...
543  33    AGAA  ATTACGAGAGGGACGAAGAGTCGCACTGCTGGACATTATACTTTGC...
769  49   GACGC  AGGAATACCATTGTGGCCCCGCACGTATTTACCTCGAAGCGCGCTC...
770  49   GACGC  AGGAGATACCATTGTGCCCGCACGTATTTACCTCGAAGCGCGCTAT...
771  49   GACGC  AGGAGATACCATTGTGCCCGCACGTATTTACCTCGAAGCGCGCTTTACC
772  49   GACGC  AGGAGATACCATTGTGCCCGCACGTATTTACCTCGAAGGGCGCTATAAC
773  49   GACGC  TGGAGATACCATTGTGCCCACGATATTTACCTCGAAGCGCGCTATTAAC
774  49   GACGC  AGGAGTTACACATTGTGCCCGCACGTATTTACCTCGAAGCGCGCTA...
775  49   GACGC  AGGAGATACCATTGTGCCCGCACTATTTACCTCGAAGCGCGCTATA...
776  49   GACGC  AGGAGATACCATTGTGCACCGCACGTATTTACCTCCGAAATGCGCT...
777  49  GACGAC  AGGAGATACCATTGTGCCCGCACGTATTTACCTCGAAGCGCGCTAT...
778  49   GATGC  AGGAGATACCATTGTGCCCGCACGTAGTTACCTCGAAGCGCGCTAT...
779  49   GACGC  AGGAATACCATTGTGGCCCGCACGGTATTTACCGTCGA

In [5]:
barcode_counts

Barcode
CTGGGGA    2
CTCTTAG    2
ATGTCGT    2
ATTCTCC    2
CAACCCA    2
          ..
GGGTGCA    1
CCACATT    1
TGCGCTC    1
AATCCGA    1
TGTGGGC    1
Name: count, Length: 978, dtype: int64

In [2]:
!pip install rapidfuzz

^C
ERROR: Operation cancelled by user
